In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import pickle

# Load data
df = pd.read_csv("/content/titanic.csv")

In [ ]:
df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [ ]:
print("Missing values per column:")
display(df.isnull().sum())

Missing values per column:


,0
pclass,0
survived,0
name,0
sex,0
age,263
sibsp,0
parch,0
ticket,0
fare,1
cabin,1014


In [ ]:
print("Class balance of the target variable 'survived':")
display(df['survived'].value_counts(normalize=True))

Class balance of the target variable 'survived':


,proportion
survived,
0,0.618029
1,0.381971


In [ ]:
X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
y = df['survived']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Redefine the preprocessing and model pipeline with correct columns for Titanic dataset
categorical_features = ['pclass', 'sex', 'embarked']
numerical_features = ['age', 'sibsp', 'parch', 'fare']

# Create preprocessing pipelines for numerical and categorical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

model_lr = Pipeline([
    ('preprocess', preprocess_lr),
    ('lr', LogisticRegression(random_state=42, solver='liblinear')) # Using liblinear solver for robustness
])

# Train the model on the training data
model_lr.fit(X_train, y_train)

# Make predictions on the test data
y_pred_lr = model_lr.predict(X_test)

# Calculate and print evaluation metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)

print(f"Logistic Regression Model Accuracy: {accuracy_lr:.4f}")
print(f"Logistic Regression Model Recall: {recall_lr:.4f}")
print(f"Logistic Regression Model Precision: {precision_lr:.4f}")

Logistic Regression Model Accuracy: 0.7748
Logistic Regression Model Recall: 0.6441
Logistic Regression Model Precision: 0.8172


In [ ]:
with open("titanic.pkl", "wb") as f:
    pickle.dump(model_eval, f)

In [ ]:
#THIS IS OUR STREAMLIT APP
%%writefile /content/app.py
import os
import pickle
import pandas as pd
import streamlit as st

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "titanic.pkl")

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

st.title("Titanic Survival Predictor")
st.write("Enter passenger details to predict survival:")

pclass = st.selectbox("Passenger Class", [1, 2, 3])
sex = st.selectbox("Sex", ['male', 'female'])
age = st.number_input("Age", 0.0, 100.0, 25.0)
sibsp = st.number_input("Number of Siblings/Spouses Aboard", 0, 10, 0)
parch = st.number_input("Number of Parents/Children Aboard", 0, 10, 0)
fare = st.number_input("Fare", 0.0, 600.0, 30.0)
embarked = st.selectbox("Port of Embarkation", ['S', 'C', 'Q'])

# Define a custom threshold (you can adjust this as needed)
SURVIVAL_THRESHOLD = 0.5
st.write("Survival Probability Threshold:", SURVIVAL_THRESHOLD)

if st.button("Predict Survival"):

    input_df = pd.DataFrame([{
        'pclass': pclass,
        'sex': sex,
        'age': age,
        'sibsp': sibsp,
        'parch': parch,
        'fare': fare,
        'embarked': embarked
    }])

    # Get probability for the positive class (survived=1)
    prob = model.predict_proba(input_df)[0][1]

    # Apply the custom threshold for prediction
    pred = 1 if prob >= SURVIVAL_THRESHOLD else 0

    label = "Survived! 🎉" if pred == 1 else "Did Not Survive 😢"

    st.subheader(label)
    st.write(f"Survival Probability: {prob:.2f}")

Writing /content/app.py
